# SFC Analysis Demo

End-to-end analysis pipeline for Spectral Flow Cytometry (SFC) data.
One cell per step — run them in order from top to bottom.

**Steps:**
1. Load an AnnData (`.h5ad` file produced by `ingest_sfc_folder()`)
2. Select a subset of events (subjects, conditions, quality gates)
3. Select relevant spectral channels (feature selection)
4. Transform and normalize the data
5. Assign a consistent color palette to conditions
6. Radar plots — per subject and per condition
7. UMAP — compute once, then plot per subject and per condition
8. PCA — compute once, then scatter and biplot

---
**Google Colab setup** — mount your Drive first:
```python
from google.colab import drive
drive.mount('/content/drive')
SFC_H5AD_PATH = '/content/drive/MyDrive/mito-marker/data/processed/sfc.h5ad'
```

**GitHub Codespaces setup** — point to the local processed file:
```python
SFC_H5AD_PATH = '../data/processed/sfc.h5ad'
```

In [ ]:
#option reingest h5ad file 
if(False):
    from mito_marker.sfc.ingestion import ingest_sfc_folder

    ingest_sfc_folder(
        data_directory_path="../tests/fixtures",
        output_file_path="../data/processed/sfc.h5ad",
    )



## Cell 1 — Load AnnData

In [ ]:
import anndata
from mito_marker.analysis import (
    select_sfc_subset,
    select_channels,
    get_selected_data_matrix,
    transform_and_normalize,
    assign_color_palette,
    plot_radar,
    compute_umap,
    plot_umap,
    compute_pca,
    plot_pca_scatter,
    plot_pca_biplot,
)

# --- Adjust this path for your environment ---
SFC_H5AD_PATH = "../data/processed/sfc.h5ad"
# Google Colab example:
# SFC_H5AD_PATH = '/content/drive/MyDrive/mito-marker/data/processed/sfc.h5ad'

print(f"Loading: {SFC_H5AD_PATH}")
sfc_anndata = anndata.read_h5ad(SFC_H5AD_PATH)
print(f"Loaded — {sfc_anndata.n_obs:,} observations × {sfc_anndata.n_vars} variables")

## Cell 2 — Interactive Subset Selection

Interactively filter the dataset by subject, condition, quality gate, and age.
Type numbers (e.g. `1,3`) or `A` for all, then press Enter.

In [ ]:
sfc_subset = select_sfc_subset(sfc_anndata)

## Cell 3 — Feature Selection (Channel Selection)

Optionally reduce the 170 spectral channels to the most informative ones.
Choose `[1] None` to keep all channels and skip this step.

Results are stored in `.var` — the AnnData shape does not change.

In [ ]:
sfc_subset = select_channels(sfc_subset)

## Cell 4 — Transformation and Normalization

Two sequential menus:
- **Step A**: Choose a transformation (None / Arcsinh / Logicle). Arcsinh with
  cofactor=150 is the cytometry standard.
- **Step B**: Choose a normalization (None / Z-score / Min-Max / L2 row / Z+L2).

The result is stored as a named `.layers` entry. Raw `.X` is never modified.

In [ ]:
sfc_subset = transform_and_normalize(sfc_subset)

## Cell 5 — Assign Color Palette

Assigns consistent hex colors to every unique condition value (dilution, diet,
FlowAI_Pass, etc.). Colors are stored in `.uns['color_palette']` and reused
automatically by all plot functions.

To override specific colors, edit `PREFERRED_CONDITION_COLORS` in
`src/mito_marker/controlled_vocabulary.py` and re-run this cell.

In [ ]:
sfc_subset = assign_color_palette(sfc_subset)

## Cell 6 — Radar Plot: Per Subject

One color per subject. Individual events appear as jittered scatter clouds;
thick lines show the group mean profile.

In [ ]:
plot_radar(sfc_subset, group_by="subject_ID")

## Cell 7 — Radar Plot: Per Condition

Replace `"dilution"` with any other filterable condition column
(e.g. `"diet"`, `"FlowAI_Pass"`, `"age"`).

In [ ]:
plot_radar(sfc_subset, group_by="dilution")

## Cell 8 — Compute UMAP (runs once, result cached)

Fits UMAP on a stratified random sample (`max_total_events=10000` by default).
Re-running this cell with the same parameters reuses cached coordinates.

To compute a second UMAP with different parameters, just run with new values —
both results coexist in `.obsm`.

In [ ]:
sfc_subset = compute_umap(sfc_subset, n_neighbors=15, min_dist=0.1)

## Cell 9 — UMAP Plot: Per Subject

In [ ]:
plot_umap(sfc_subset, group_by="subject_ID")

## Cell 10 — UMAP Plot: Per Condition

In [ ]:
plot_umap(sfc_subset, group_by="dilution")

## Cell 11 — Compute PCA (runs once, result cached)

Fits PCA on ALL events (much faster than UMAP — no subsampling needed).
Coordinates, loadings, and explained variance are all stored in `.obsm` / `.uns`.

In [ ]:
sfc_subset = compute_pca(sfc_subset, n_components=50)

## Cell 12 — PCA Scatter: Per Subject

PC1 vs PC2 scatter with centroids marked as `X`. Axis labels show the
explained variance percentage.

In [ ]:
plot_pca_scatter(sfc_subset, group_by="subject_ID")

## Cell 13 — PCA Scatter: Per Condition

In [ ]:
plot_pca_scatter(sfc_subset, group_by="dilution")

## Cell 14 — PCA Biplot: Per Subject

Correlation circle with loading arrows (gray) and group centroids. Channels
with high loading magnitude are labeled in dark red.

The top 5 channels per axis are also printed to the console before the figure.

In [ ]:
plot_pca_biplot(sfc_subset, group_by="subject_ID", top_n_variables=5)

## Cell 15 — PCA Biplot: Per Condition

In [ ]:
plot_pca_biplot(sfc_subset, group_by="dilution", top_n_variables=5)